# Compare two models

`orinth.models.predictor()` returns the object `/inference` runs, so what you
measure here is exactly what the platform will do.

The number worth having is not "which scored higher" — `orinth.evaluate` answers
that, and records it. It is **where they disagree**: two models at the same
accuracy can be right about different things, and that is an ensemble, or a bug.

In [ ]:
import orinth

for model in orinth.models.list():
    print(f"{model.id:42} {model.family:22} {model.task_type:20} available={model.available}")

In [ ]:
TASK = "classification"


def loads(model_id):
    # `available` is a filesystem check: the artifact is where the registry says
    # it is. Whether it *deserializes* is a different question — an old export
    # can fail on a Keras version bump — and the honest way to find out is to
    # try, once, before building a comparison around it.
    try:
        orinth.models.predictor(model_id)
        return True
    except Exception as error:
        print(f"skipping {model_id}: {type(error).__name__}")
        return False


pair = [
    model.id
    for model in orinth.models.list()
    if model.available and model.task_type == TASK and loads(model.id)
][:2]
A, B = pair

DATASET_ID = next(
    dataset.id
    for dataset in orinth.datasets.list()
    if dataset.task_type == TASK and dataset.splits.get("valid")
)
print("comparing\n ", A, "\n ", B, "\non", DATASET_ID)

The first call to each predictor loads weights and is slow; both are cached
afterwards, so re-running the next cell costs only inference.

In [ ]:
from pathlib import Path

predictor_a = orinth.models.predictor(A)
predictor_b = orinth.models.predictor(B)
params = orinth.models.parameters()

frame = orinth.datasets.load(DATASET_ID, "valid", limit=60)


def verdict(predictor, path):
    scores = predictor.classify(path, params)
    if scores:
        return max(scores, key=scores.get), float(max(scores.values()))
    detections = predictor.predict(path, params)
    return orinth.models.image_label(detections), max(
        (float(d.confidence) for d in detections), default=0.0
    )


rows = []
for row in frame.iter_rows(named=True):
    path = Path(row["path"])
    label_a, confidence_a = verdict(predictor_a, path)
    label_b, confidence_b = verdict(predictor_b, path)
    rows.append(
        {
            "item": row["item_id"],
            "truth": row["label"] or "Normal",
            "a": label_a,
            "b": label_b,
            "conf_a": round(confidence_a, 3),
            "conf_b": round(confidence_b, 3),
        }
    )

print(len(rows), "items scored by both")

## Agreement, and who is right when they disagree

Four buckets. The two that matter are the diagonal ones: where A is right and B
is wrong, and the reverse. If both are large, neither model dominates and the
question changes from "which one" to "what makes them differ".

In [ ]:
import polars as pl

scored = pl.DataFrame(rows).with_columns(
    (pl.col("a") == pl.col("truth")).alias("a_right"),
    (pl.col("b") == pl.col("truth")).alias("b_right"),
    (pl.col("a") == pl.col("b")).alias("agree"),
)

print(f"accuracy  A={scored['a_right'].mean():.3f}  B={scored['b_right'].mean():.3f}")
print(f"agreement {scored['agree'].mean():.3f}\n")

scored.group_by(["a_right", "b_right"]).agg(pl.len().alias("items")).sort("items", descending=True)

## The disagreements themselves

Sorted by how confident the wrong one was. A model that is confidently wrong
where the other is right is the most informative row in the whole comparison.

In [ ]:
scored.filter(~pl.col("agree")).select(
    ["item", "truth", "a", "conf_a", "b", "conf_b"]
).head(12)

## Log the comparison

Runs are charted on this notebook's page. For the recorded, side-by-side version
that the Testing page can rank, use `orinth.evaluate.compare([A, B], key)`.

In [ ]:
with orinth.runs.start("model comparison", params={"a": A, "b": B}, dataset_id=DATASET_ID) as run:
    run.log(
        accuracy_a=float(scored["a_right"].mean()),
        accuracy_b=float(scored["b_right"].mean()),
        agreement=float(scored["agree"].mean()),
    )
    run.log_text("verdict", f"{A} vs {B} on {DATASET_ID}/valid, {scored.height} items")
    run.log_artifact(scored.write_csv().encode("utf-8"), name="comparison.csv")